In [0]:
import time, random
from datetime import datetime, timezone, timedelta


In [0]:
%py
### Generate random streams

def generate_stream(
    path: str,
    column_name: str,
    low: float,
    high: float,
    device_count: int,
    batch_interval_s: int,
    latency_max_s: int
):
    """
    Generic IoT stream generator:
    - Emits `device_count` rows each `batch_interval_s` seconds.
    - Each row has:
        - `device_id`: random integer between 1 and `device_count`
        - `column_name`: random float between `low` and `high`
        - event_time: now - random ( 0 to `latency_max_s` seconds)
        - append into Delta path or Delta table `path`
    """

    while True:
        now = datetime.now(timezone.utc)
        data = []
        for device_id in range(1, device_count + 1):
            ts = now - timedelta(seconds=random.uniform(0, latency_max_s))

            value = round(random.uniform(low, high),4)

            data.append({
                "device_id": device_id,
                "event_time": ts,
                column_name: value
            })

        df = spark.createDataFrame(data)
        df.write.format("delta").mode("append").save(path)
        
        time.sleep(batch_interval_s)


In [0]:
#launch all three generators concurently without a wrapper

from concurrent.futures import ThreadPoolExecutor

# assume generate_stream is already defined above

device_count = 5 # five bridges
batch_interval_s = 60 # emit every 60 seconds
latency_max_s = 60 # max latency of 60 seconds

#(path, column, low, high)
streams = [
    ("/Volumes/bridge_monitoring/00_landing/streaming/bridge_temperature", "temperature", 10, 30),
    ("/Volumes/bridge_monitoring/00_landing/streaming/bridge_vibration", "vibration", 0.005, 0.05),
    ("/Volumes/bridge_monitoring/00_landing/streaming/bridge_tilt", "tilt_angle", -0.005, 0.005)
]

with ThreadPoolExecutor(max_workers=len(streams)) as executor:
    for path, column_name, low, high in streams:
        executor.submit(
            generate_stream,
            path,
            column_name,
            low,
            high,
            device_count,
            batch_interval_s,
            latency_max_s
        )
    #context manager will call shutdown (wait=True) here,
    executor.shutdown(wait=True)
    # and block forever because these tasks never return

